In [10]:
%pip -q install "tokenizers<0.14" "huggingface_hub<0.24" faster-whisper jiwer pandas
# Make sure ffmpeg is installed locally (e.g. `brew install ffmpeg` on macOS)
import subprocess, shutil
if not shutil.which("ffmpeg"):
    print("WARNING: ffmpeg not found. Install it: brew install ffmpeg (macOS) or apt install ffmpeg (Linux)")
else:
    print("ffmpeg found.")
print("Dependencies ready.")

^C
  File "/usr/local/anaconda3/lib/python3.8/site-packages/pip/_internal/cli/progress_bars.py", line 12, in <module>
    from pip._internal.utils.logging import get_indentation
  File "/usr/local/anaconda3/lib/python3.8/site-packages/pip/_internal/utils/logging.py", line 18, in <module>
    from pip._internal.utils.misc import ensure_dir
  File "/usr/local/anaconda3/lib/python3.8/site-packages/pip/_internal/utils/misc.py", line 21, in <module>
    from pip._vendor import pkg_resources
  File "/usr/local/anaconda3/lib/python3.8/site-packages/pip/_vendor/pkg_resources/__init__.py", line 3252, in <module>
    def _initialize_master_working_set():
  File "/usr/local/anaconda3/lib/python3.8/site-packages/pip/_vendor/pkg_resources/__init__.py", line 3235, in _call_aside
    f(*args, **kwargs)
  File "/usr/local/anaconda3/lib/python3.8/site-packages/pip/_vendor/pkg_resources/__init__.py", line 3264, in _initialize_master_working_set
    working_set = WorkingSet._build_master()
  File "/usr/l

In [ ]:
import os
import re
import time
import pandas as pd
from faster_whisper import WhisperModel
from IPython.display import display

AUDIO_DIR = "./audio"
os.makedirs(AUDIO_DIR, exist_ok=True)

_model_cache = {}

def get_model(name="base"):
    if name not in _model_cache:
        print(f"Loading faster-whisper model '{name}'...")
        _model_cache[name] = WhisperModel(name, device="cpu", compute_type="int8")
    return _model_cache[name]

def normalize_text(text):
    text = str(text).lower().strip()
    text = re.sub(r"[^\w\s]", "", text)
    return text

def read_csv_auto(path):
    for enc in ["utf-8", "utf-8-sig", "latin-1"]:
        try:
            return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError:
            continue
    return pd.read_csv(path, encoding="latin-1")

print("Imports and helpers ready.")

Imports and helpers ready.


In [ ]:
INTENT_CONFIG = {
    "decision": {
        "tower.decision.right": [["right"], ["2"], ["two"], ["second"]],
        "tower.decision.left": [["left"], ["1"], ["number", "one"], ["first"]],
        "tower.decision.same": [["both"], ["together"], ["same", "time"], ["one", "and", "two"]]
    },
    "reason": {
        "tower.reason.tall": [["tall"], ["taller"], ["big"], ["bigger"], ["high"], ["higher"]],
        "tower.reason.weight": [["weight"], ["weighs"], ["top"], ["heavy"], ["heavier"]],
        "tower.reason.thinnerbase": [["thin"], ["thinner"], ["small"], ["smaller"], ["base"], ["bottom"]],
        "tower.reason.notsymmetrical": [["symmetry"], ["symmetric"], ["symmetrical"], ["both", "sides"], ["not", "the", "same"], ["asymmetrical"], ["asymmetric"], ["not", "same"]]
    }
}

def determine_intent(transcript, state):
    if not transcript or str(transcript).strip() == "":
        return "default.noaudio"

    clean = normalize_text(transcript)
    words = clean.split()

    state_intents = INTENT_CONFIG.get(state, {})

    for intent_name, groups in state_intents.items():
        for group in groups:
            if all(any(re.fullmatch(re.escape(k), w) for w in words) for k in group):
                return intent_name

    return "default.unknown"

In [ ]:
INPUT_CSV = "./reason_prediction_audio_log.csv"
WHISPER_MODEL_BATCH = "base"
BATCH_STATE = "default"

KEYWORDS = {
    "reason": [
        "tall", "taller", "big", "high", "higher", "weight", "weighs", "top",
        "heavy", "heavier", "thin", "thinner", "small", "smaller",
        "symmetric", "symmetrical", "both sides", "asymmetrical", "asymmetric"
    ],
    "decision": [
        "right", "two", "left", "one", "both", "together", "same time",
    ],
    "default": [
        "tall", "taller", "big", "high", "higher", "weight", "weighs", "top",
        "heavy", "heavier", "thin", "thinner", "small", "smaller",
        "symmetric", "symmetrical", "both sides", "asymmetrical", "asymmetric",
        "right", "two", "left", "one", "both", "together", "same time",
    ],
}

def run_batch_pipeline(csv_path, model_name, state):
    if not os.path.exists(csv_path):
        print(f"Error: CSV not found at {csv_path}")
        print(f"Place your CSV file at: {os.path.abspath(csv_path)}")
        return

    df = read_csv_auto(csv_path)

    transcript_col = "re-run transcription"
    intent_col = f"re-run intent [{state}]"
    timing_col = "re-run processing time [ms]"

    for col in [transcript_col, intent_col, timing_col]:
        if col not in df.columns:
            df[col] = ""

    model = get_model(model_name)
    processed_count = 0
    skipped_count = 0
    error_count = 0

    keywords = KEYWORDS.get(state, KEYWORDS["default"])
    batch_prompt = "Try to focus specifically on these words as you listen to the audio: " + ", ".join(keywords)
    print(f"INITIAL PROMPT [{state}]: {batch_prompt}")

    print(f"Starting batch process for state: {state} on {len(df)} rows...\n")

    for idx, row in df.iterrows():
        if str(row.get(intent_col, '')).strip() not in ["", "nan"]:
            continue

        file_name = str(row['fileName']).strip()
        wav_path = os.path.join(AUDIO_DIR, file_name)

        if not os.path.exists(wav_path):
            skipped_count += 1
            continue

        existing_text = str(row.get(transcript_col, '')).strip()
        if existing_text in ["", "nan"]:
            try:
                start_time = time.time()
                segments, info = model.transcribe(wav_path, initial_prompt=batch_prompt, language="en")
                new_text = " ".join(seg.text.strip() for seg in segments).strip()
                elapsed_ms = (time.time() - start_time) * 1000

                df.at[idx, transcript_col] = new_text
                df.at[idx, timing_col] = round(elapsed_ms, 2)
            except Exception as e:
                print(f"Row {idx} ERROR: {file_name} — {e}")
                df.at[idx, transcript_col] = "ERROR"
                error_count += 1
                continue
        else:
            new_text = existing_text

        new_intent = determine_intent(new_text, state)

        df.at[idx, intent_col] = new_intent
        processed_count += 1
        print(f"Row {idx} processed: {file_name}")

    df.to_csv(csv_path, index=False)
    print(f"\nBatch Finished!")
    print(f"Processed: {processed_count} | Skipped: {skipped_count} | Errors: {error_count}")
    display(df.head(10))

run_batch_pipeline(INPUT_CSV, WHISPER_MODEL_BATCH, BATCH_STATE)

Loading faster-whisper model 'base'...
INITIAL PROMPT [default]: Try to focus specifically on these words as you listen to the audio: tall, taller, big, high, higher, weight, weighs, top, heavy, heavier, thin, thinner, small, smaller, symmetric, symmetrical, both sides, asymmetrical, asymmetric, right, two, left, one, both, together, same time
Starting batch process for state: default on 124 rows...

Row 0 processed: 2026-6-14--10-14--3.wav
Row 1 processed: 2026-6-14--10-14--4.wav
Row 2 processed: 2026-6-14--10-14--5.wav


KeyboardInterrupt: 

In [ ]:
def run_intent_classification(csv_path):
    if not os.path.exists(csv_path):
        print(f"Error: CSV not found at {csv_path}")
        return

    df = read_csv_auto(csv_path)

    transcript_col = "re-run transcription"
    decision_col = "re-run intent [decision]"
    reason_col = "re-run intent [reason]"

    if transcript_col not in df.columns:
        print(f"No '{transcript_col}' column found.")
        return

    for col in [decision_col, reason_col]:
        if col not in df.columns:
            df[col] = ""

    classified = 0
    for idx, row in df.iterrows():
        transcript = str(row.get(transcript_col, '')).strip()
        if transcript in ["", "nan", "ERROR"]:
            continue

        df.at[idx, decision_col] = determine_intent(transcript, "decision")
        df.at[idx, reason_col] = determine_intent(transcript, "reason")
        classified += 1

    df.to_csv(csv_path, index=False)
    print(f"Classified {classified} rows for both decision and reason intents.")
    display(df[[transcript_col, decision_col, reason_col]].head(20))

run_intent_classification(INPUT_CSV)

Classified 119 rows for both decision and reason intents.


,re-run transcription,re-run intent [decision],re-run intent [reason]
0,ive had a cut up in our base ive had a cut up ...,default.unknown,tower.reason.thinnerbase
1,and the head of inner base and the head of inn...,default.unknown,tower.reason.thinnerbase
2,"together, same, same, same, same, same",tower.decision.same,default.unknown
3,"ive, you can't be kept in half more weight, so...",default.unknown,tower.reason.weight
4,it doesn't have a thinner base.,default.unknown,tower.reason.thinnerbase
5,"Yeah, in that case, we have to make it a littl...",default.unknown,tower.reason.thinnerbase
6,"肯定, because that's now where it was owner and ...",default.unknown,default.unknown
7,ingred that has a smaller base.,default.unknown,tower.reason.thinnerbase
8,"up, it's not the same on both sides",tower.decision.same,tower.reason.notsymmetrical
9,"ive, the bottom is, the body straight.",default.unknown,tower.reason.thinnerbase
